In [1]:
# Cell 1 - Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
scaler = pickle.load(open('diamond_scaler.sav', 'rb'))
model  = pickle.load(open('diamond_model.sav', 'rb'))

In [3]:
df = pd.read_csv('DiamondPrices2.csv').set_index('id')
df.head()

,carat,cut,color,clarity,depth,table,x,y,z
id,,,,,,,,,
1,0.23,Ideal,E,SI2,61.5,55.0,3.95,3.98,2.43
2,0.21,Premium,E,SI1,59.8,61.0,3.89,3.84,2.31
3,NaN,Good,E,VS1,56.9,65.0,4.05,4.07,2.31
4,0.29,Premium,I,VS2,62.4,58.0,4.20,4.23,2.63
5,0.31,Good,J,SI2,63.3,58.0,4.34,4.35,2.75


In [4]:
df.describe()

,carat,depth,table,x,y,z
count,51210.000000,51262.000000,51254.000000,51209.000000,51274.000000,51284.000000
mean,0.797648,61.749467,57.461613,5.733212,5.734110,3.538269
std,0.474164,1.431262,2.234639,1.121856,1.144095,0.706337
min,0.200000,43.000000,43.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,4.720000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,5.700000,5.710000,3.520000
75%,1.040000,62.500000,59.000000,6.540000,6.540000,4.030000
max,5.010000,79.000000,95.000000,10.740000,58.900000,31.800000


In [5]:
df.columns

Index(['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z'], dtype='str')

In [6]:
df = df[(df['x'] > 0) & (df['y'] > 0) & (df['z'] > 0)]
#remove rows where x = 0 , y = 0 ,z = 0 and nan

In [7]:
# Remove outliers using percentiles
for col in ['x', 'y', 'z']:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df = df[(df[col] >= lower) & (df[col] <= upper)]

In [8]:
df.drop_duplicates(keep='first', inplace = True)

In [9]:
df['cut'] = df['cut'].fillna(df['cut'].mode()[0])
df['cut'] = df['cut'].astype('category')
#fill empty value with mode since its a categorical value

In [10]:
df['color'] = df['color'].fillna(df['color'].mode()[0])
df['color'] = df['color'].astype('category')
#fill empty value with mode since its a categorical value

In [11]:
df['clarity'] = df['clarity'].fillna(df['clarity'].mode()[0])
df['clarity'] = df['clarity'].astype('category')

In [12]:
df['depth'] = df['depth'].fillna(df['depth'].mean())

In [13]:
df['carat'] = df['carat'].fillna(df['carat'].median())

In [14]:
df['carat_log'] = np.log1p(df['carat'])

In [15]:

df['table'] = df['table'].fillna(df['table'].median())
#fill empty value with median since its a numerical value

df['table_log'] = np.log1p(df['table'])
#deal with outlier

In [16]:
dfcate = pd.get_dummies(df[['color','cut','clarity']], drop_first=True).astype(int)

In [17]:
scalednumericaldf = df[['carat_log','depth','table_log','x' ,'y' ,'z' ]]


In [18]:
scalednumericaldf = pd.DataFrame(scaler.transform(scalednumericaldf), columns=['carat_log','depth','table_log','x' ,'y' ,'z' ], index=df.index)

In [19]:
expected_cols = model.feature_names_in_


In [20]:
# Reindex to match training columns (missing cols become 0)
trainingset = pd.concat([scalednumericaldf ,dfcate], axis=1)
trainingset = trainingset.reindex(columns=model.feature_names_in_, fill_value=0)

In [21]:
print(trainingset.columns.tolist())
print(model.feature_names_in_)
#checking that training df allign with model

['carat_log', 'depth', 'table_log', 'x', 'y', 'z', 'color_E', 'color_F', 'color_G', 'color_H', 'color_I', 'color_J', 'cut_Good', 'cut_Ideal', 'cut_Premium', 'cut_Very Good', 'clarity_IF', 'clarity_SI1', 'clarity_SI2', 'clarity_VS1', 'clarity_VS2', 'clarity_VVS1', 'clarity_VVS2']
['carat_log' 'depth' 'table_log' 'x' 'y' 'z' 'color_E' 'color_F' 'color_G'
 'color_H' 'color_I' 'color_J' 'cut_Good' 'cut_Ideal' 'cut_Premium'
 'cut_Very Good' 'clarity_IF' 'clarity_SI1' 'clarity_SI2' 'clarity_VS1'
 'clarity_VS2' 'clarity_VVS1' 'clarity_VVS2']


In [22]:
predicted = np.expm1(model.predict(trainingset))
predicted = pd.DataFrame({'PredictedPrice': predicted},index=trainingset.index)
#predicting model

In [23]:
predict = pd.concat([df,predicted], axis = 1)
predict.describe()
#combining predicted price and df

,carat,depth,table,x,y,z,carat_log,table_log,PredictedPrice
count,43498.000000,43498.000000,43498.000000,43498.000000,43498.000000,43498.000000,43498.000000,43498.000000,43498.000000
mean,0.768779,61.757593,57.415143,5.704171,5.706209,3.522802,0.546224,4.066899,3747.671257
std,0.400940,1.359090,2.165570,1.021631,1.015143,0.629679,0.216360,0.036628,3904.190174
min,0.250000,43.000000,43.000000,4.100000,4.160000,2.610000,0.223144,3.784190,325.631968
25%,0.410000,61.100000,56.000000,4.750000,4.750000,2.930000,0.343590,4.043051,1016.493033
50%,0.700000,61.800000,57.000000,5.700000,5.710000,3.530000,0.530628,4.060443,2296.901618
75%,1.020000,62.500000,59.000000,6.510000,6.500000,4.020000,0.703098,4.094345,4975.993980
max,2.100000,79.000000,79.000000,8.340000,8.170000,5.030000,1.131402,4.382027,29280.555561


In [24]:
predict.to_csv('predict_pr.csv',index=False)

In [25]:
p1predicted = pd.read_csv('DiamondPrices.csv').set_index('id')[['price']]
p1predicted , predicted = p1predicted.align(predicted , join = 'inner', axis = 0 )
#import price from phase 1
#align df from p1 and p2

r2 = r2_score(p1predicted['price'], predicted['PredictedPrice'])
mae = mean_absolute_error(p1predicted['price'], predicted['PredictedPrice'])
print("R²:", r2)
print("MAE:", mae)
#calculate R² value
#calculate Mean Average Error

R²: 0.9140611976497612
MAE: 500.23813526121825
